In [1]:
## General imports
from multiprocessing import Manager, Pool, cpu_count
from functools import partial
## Local imports
import get_data_matrix as getMat
import mapLearningUtils as mlu
import train_MappingJM as trMap

In [2]:
## Pre-processing parameters
savePath      = './all_params/folded_ablated_data/'
ablationsList = ['velocity', 'anthropo', 'forcestorques', 'forces', 'torques', 'Fx', 'Fy', 'Fz', 'Tx', 'Ty', 'Tz']
foldsList     = ['2', '5']

## Mapping parameters
# Common
dataDir         = './all_params/folded_ablated_data/'
fittedModelsDir = './all_params/all_fitted_models/'
fitTypesList    = ['MVLR', 'MVPR'] # FNO, LSTM

# MVPR
degreesList = [2, 3] # Degree of the fitted polynomial for each input

# FNO
nbModesList   = [4, 8, 16, 32]       # Number of modes [number of low Fourier frequencies]
nbHChanList   = [8, 16, 32, 64] # Number of hidden channels [dimensionnality of the network's internal features]
nbMaxIterList = [100, 250, 500]      # Maximum number of training iterations
limLoss       = 1e-6                 # Convergence limit (if less improvement between two iterations then training stops)

# LSTM
# TODO

In [3]:
## Prepare lists of parameters dictionnaries to train different versions of the models
list_params_sets = []
for fitting in fitTypesList:
    for ablation in ablationsList:
        for fold in foldsList:
            fold_name = fold + '-fold'
            data_path = dataDir + ablation + '_' + fold_name + '.pkl'
            if fitting == 'MVLR':
                saveDirFile = fittedModelsDir + fitting + '_' + ablation + '_' + fold_name + '.pkl'
                dict_params = {'model': fitting, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold),
                               'dataPath': data_path, 'fittedModelDir': fittedModelsDir, 'saveFittedDir': saveDirFile}
                list_params_sets.append(dict_params)
            elif fitting == 'MVPR':
                for degree in degreesList:
                    saveDirFile = fittedModelsDir + fitting + '_' + ablation + '_' + fold_name + '_deg' + str(degree) + '.pkl'
                    dict_params = {'model': fitting, 'degree': degree, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold),
                                   'dataPath': data_path, 'fittedModelDir': fittedModelsDir, 'saveFittedDir': saveDirFile}
                    list_params_sets.append(dict_params)
            elif fitting == 'FNO':
                "FNO is not yet handled: Needs HPC"
            else:
                "The requested fitting type is not yet handled."

In [6]:
import pickle
test_path = './all_params/folded_ablated_data/anthropo_2-fold.pkl'

with open(test_path, 'rb') as file:
    data = pickle.load(file)

data.get('fold1')

{'train_subjs_calib': ['S8_c',
  'S1_c',
  'S15_c',
  'S13_c',
  'S11_c',
  'S9_c',
  'S16_c',
  'S12_c'],
 'eval_subjs_calib': array(['S10_c', 'S6_c', 'S7_c', 'S5_c', 'S4_c', 'S2_c', 'S17_c', 'S14_c',
        'S3_c'], dtype=object),
 'train_subjs_assist': ['S04_a', 'S02_a', 'S10_a', 'S06_a', 'S05_a'],
 'eval_subjs_assist': array(['S09_a', 'S01_a', 'S12_a', 'S14_a', 'S03_a'], dtype=object),
 'train_data': array([[-0.5354035110028602, -0.41580391002573613, 5.908805828269533,
         ..., 5.4793905238252815e-05, 0, 'S1_c'],
        [-0.5297526197729584, -0.42156316488698914, 6.123443624227066,
         ..., -0.001982414362333865, 0, 'S1_c'],
        [-0.5240278009357009, -0.42704510857260336, 6.26648198657429, ...,
         -0.003993995859390828, 0, 'S1_c'],
        ...,
        [-0.054625685837181526, -0.38260709933063486, 8.118446650391121,
         ..., 0.0018708099350917828, 25, 'S10_a'],
        [-0.047261998191328494, -0.38376446763535926, 8.059369433879725,
         ..., 0.001785